In [2]:
import numpy as np
import pandas as pd
from nba_api.stats.static import players
from nba_api.stats.endpoints import commonplayerinfo
from nba_api.stats.endpoints import playerindex

# **Data Cleaning Pipeline**

## Step 1: Datasets Merging, Alignment & Verification Pipeline

This step builds a unified shot tracking matrix by combining two highly different datasets. Because these files come from separate data vendors, we execute a multi-layered engineering pipeline to ensure a strict 1-to-1 match without data anomalies.

### The Component Datasets
1.  **`df1` (Official Play-by-Play Shot Logs):** Contains standard descriptive event fields (`ACTION_TYPE`, `SHOT_TYPE`, spatial coordinates `LOC_X` / `LOC_Y`) and the official outcome flag (`SHOT_MADE_FLAG`).
2.  **`df2` (Advanced Tracking Logs):** Contains high-fidelity player tracking attributes (`CLOSEST_DEFENDER`, `CLOSE_DEF_DIST`, physical tracking context like `TOUCH_TIME` and `DRIBBLES`), and the outcome string (`SHOT_RESULT`).

---

### Pipeline Execution Flow

The data engine processes the merging sequences in a strict, chronological flow:

#### 1. Data Loading (`pd.read_csv`)
Imports the official play-by-play logs (`df1`) and advanced tracking tables (`df2`) into the environment.

#### 2. Time Linearization (`MATCH_TIME`)
Normalizes game clocks across both datasets. Because one logs remaining period time and the other logs active possession clocks, we convert all timestamps into a continuous scalar: **total absolute seconds elapsed from the opening tip-off**.

#### 3. Type Sanitization & Guarding
Standardizes key columns by converting `GAME_ID` and `PLAYER_ID` to trimmed string formats (`.astype(str).str.strip()`), preventing whitespace mismatches from causing silent join drops.

#### 4. Core Logical Inner Join (Strict Constraints)
We pair data arrays using a multi-key validation check:
*   **Entity Linking:** Rows must share the exact same `GAME_ID` and `PLAYER_ID`.
*   **Outcome Identity Integration:** The shot results *must match perfectly* on both sides. A binary `1` (make) in `df1` is mapped against string values `made` in `df2`. If one dataset says a shot was made and the other says it was missed, the pair is immediately discarded.

#### 5. Smart Scoring Matrix Execution (Temporal Window & Penalties)
Because data vendors suffer from minor **sub-second clock drift**, we apply a **maximum 5-second difference threshold (`TIME_DIFF <= 5`)**. Within this 5-second cluster window, we run a semantic-physical scoring algorithm to eliminate false matches:

$$ Smart\_Rank\_Score = (5 - Time\_Diff) - Physical\_Penalty $$

*   **Baseline Proximity Score (`5 - TIME_DIFF`):** Closer timestamps score higher. A 0-second gap scores +5; a 3-second gap scores +2.
*   **Physical Rule Violations (`PHYSICAL_PENALTY`):** If tracking metrics violate the physical laws of the play-by-play description, a heavy **20-point penalty** is applied:
    *   *The Tip-In Violation:* Official description says `Tip-in` / `Putback`, but tracking claims `TOUCH_TIME > 1.5s` or `DRIBBLES > 0` $A \rightarrow B$ **Penalty + 20.0** (Demoted).
    *   *The Drive Violation:* Official description says `Driving` / `Pull-up`, but tracking records a catch-and-shoot with zero separation (`TOUCH_TIME <= 1.2s` and `DRIBBLES == 0`) $A \rightarrow B$ **Penalty + 20.0** (Demoted).

#### 6. Bidirectional Deduplication
The unified matrix is sorted by `SMART_RANK_SCORE` in descending order (pushing the most logically sound records to the top). We then execute a bidirectional `.drop_duplicates()` filter across both `MATCH_TIME_df1` and `MATCH_TIME_df2`. This safely enforces a mathematically sound **1-to-1 event mapping**.

In [ ]:
df1 = pd.read_csv('data/nba_api_2014-15_shots.csv')
df2 = pd.read_csv('data/shot_logs(in).csv')

def get_total_elapsed_seconds_df1(row):
    period_duration = 720 if row['PERIOD'] <= 4 else 300
    secs_remaining = int(row['MINUTES_REMAINING']) * 60 + int(row['SECONDS_REMAINING'])
    secs_elapsed = period_duration - secs_remaining
    total_elapsed = sum([720 if p <= 4 else 300 for p in range(1, row['PERIOD'])]) + secs_elapsed
    return total_elapsed

def get_total_elapsed_seconds_df2(row):
    period_duration = 720 if row['PERIOD'] <= 4 else 300
    try:
        parts = str(row['GAME_CLOCK']).split(':')
        secs_remaining = int(parts[0]) * 60 + int(parts[1])
    except:
        secs_remaining = int(float(row['GAME_CLOCK']))
    secs_elapsed = period_duration - secs_remaining
    total_elapsed = sum([720 if p <= 4 else 300 for p in range(1, row['PERIOD'])]) + secs_elapsed
    return total_elapsed

df1['MATCH_TIME'] = df1.apply(get_total_elapsed_seconds_df1, axis=1)
df2['MATCH_TIME'] = df2.apply(get_total_elapsed_seconds_df2, axis=1)


df1['GAME_ID'] = df1['GAME_ID'].astype(str).str.strip()
df1['PLAYER_ID'] = df1['PLAYER_ID'].astype(str).str.strip()
df2['GAME_ID'] = df2['GAME_ID'].astype(str).str.strip()
df2['PLAYER_ID'] = df2['player_id'].astype(str).str.strip()


merged_df = pd.merge(df1, df2, on=['GAME_ID', 'PLAYER_ID'], suffixes=('_df1', '_df2'))

merged_df['TIME_DIFF'] = (merged_df['MATCH_TIME_df1'] - merged_df['MATCH_TIME_df2']).abs()
merged_df['DIST_DIFF'] = (merged_df['SHOT_DISTANCE'] - merged_df['SHOT_DIST']).abs()

df2_made_mapped = merged_df['SHOT_RESULT'].astype(str).str.lower().str.strip() == 'made'
merged_df['SHOT_MATCH_VALID'] = merged_df['SHOT_MADE_FLAG'].astype(int) == df2_made_mapped.astype(int)



# Smart Semantic-Physical Cross-Ranking Engine
action_lower = merged_df['ACTION_TYPE'].astype(str).str.lower()
merged_df['PHYSICAL_PENALTY'] = 0.0

# [Defense Line A]: If official log states a tip-in/putback, but tracking data 
# shows extended possession (>1.5s) or active dribbling, apply a heavy penalty (+20.0).
is_tip_action = action_lower.str.contains('tip|putback')
bad_tip_match = is_tip_action & ((merged_df['TOUCH_TIME'] > 1.5) | (merged_df['DRIBBLES'] > 0))
merged_df.loc[bad_tip_match, 'PHYSICAL_PENALTY'] += 20.0

# [Defense Line B]: If official log states an aggressive drive/on-ball play, but tracking 
# data shows zero dribbles and ultra-short possession (<=1.2s), apply a heavy penalty (+20.0).
drive_keywords = ['driving', 'pullup', 'step', 'turnaround', 'fadeaway', 'running']
is_drive_action = action_lower.str.contains('|'.join(drive_keywords))
bad_drive_match = is_drive_action & (merged_df['TOUCH_TIME'] <= 1.2) & (merged_df['DRIBBLES'] == 0)
merged_df.loc[bad_drive_match, 'PHYSICAL_PENALTY'] += 20.0

# Calculate the final Smart Ranking Score
merged_df['SMART_RANK_SCORE'] = (5 - merged_df['TIME_DIFF']) - merged_df['PHYSICAL_PENALTY']



valid_mask = (
    (merged_df['TIME_DIFF'] <= 5) & 
    (merged_df['SHOT_MATCH_VALID']) & 
    (merged_df['DIST_DIFF'] <= 3)
)
final_merged_df = merged_df[valid_mask].copy()

final_merged_df = final_merged_df.sort_values('SMART_RANK_SCORE', ascending=False)

# 1-to-1 event mapping
final_merged_df = final_merged_df.drop_duplicates(subset=['GAME_ID', 'PLAYER_ID', 'MATCH_TIME_df1'], keep='first')
final_merged_df = final_merged_df.drop_duplicates(subset=['GAME_ID', 'PLAYER_ID', 'MATCH_TIME_df2'], keep='first')


## Step 2: Feature Engineering & Biographical Data Integration

We enrich matched shot logs with physical attributes via `nba_api` and engineer continuous features for ML models:

*   **API Data Retrieval:** Fetches player profiles using `PlayerIndex` and scrubs `NaN` values.
*   **Height Standardization (`HEIGHT_INCHES`):** Converts text heights (e.g., `"6-10"`) into continuous scalar integers.
*   **Temporal Compaction (`PERIOD_TIME_REMAINING`):** Flattens minutes and seconds into a single continuous feature of remaining seconds.
*   **Database Enrichment:** Integrates engineered traits into logs via a `PLAYER_ID` left join.


In [ ]:
all_players_data = playerindex.PlayerIndex(historical_nullable=1).get_data_frames()[0]
#all_players_data = playerindex.PlayerIndex(season="2014-15").get_data_frames()[0]

player_detail = all_players_data[['PERSON_ID', 'HEIGHT', 'WEIGHT']]
player_detail.columns = ['PLAYER_ID', 'HEIGHT', 'WEIGHT']
player_detail = player_detail.dropna(subset=['HEIGHT', 'WEIGHT'])

#Split the Height column into ft and in
split_height = player_detail['HEIGHT'].str.split('-', expand=True).astype(int)
player_detail['HEIGHT_INCHES'] = (split_height[0]*12) + split_height[1]


player_detail['PLAYER_ID'] = player_detail['PLAYER_ID'].astype(str).str.strip()
final_merged_df['PLAYER_ID'] = final_merged_df['PLAYER_ID'].astype(str).str.strip()

final_merged_df = pd.merge(final_merged_df, player_detail, on='PLAYER_ID', how='left')
final_merged_df['PERIOD_TIME_REMAINING'] = (final_merged_df['MINUTES_REMAINING'].astype(int)*60) + final_merged_df['SECONDS_REMAINING'].astype(int)

## Step 3: Matrix Pruning & Quality Assurance

We filter the dataset to target features, clean naming conventions, and run a data quality audit to eliminate corrupted logs before modeling.

* **Dimensionality Reduction**: Removed redundant diagnostic tracking features, keeping only high-signal markers.
* **Structural Refactoring**: Renamed technical merge labels (e.g., `PERIOD_df1`, `MATCH_TIME_df1`) to clean variables (`PERIOD`, `MATCH_TIME`).
* **Quality Assurance Audits**:
  * **Possession Time Audit**: Filtered out **3,358 records with `TOUCH_TIME == 0`** and **312 records with `TOUCH_TIME < 0`** to eliminate these physically impossible tracking anomalies.
  * **Missing Values**: Executed `.dropna()` to clear remaining missing values (NaN) to protect model stability.
* **Final Preprocessed Volume**: The final cleaned dataset contains **95,892 clean shot rows** saved directly to `data/shotlog_2014-15_preprocessed.csv`.


In [ ]:
columns_to_keep = [
    # IDs & Names
    'GAME_ID', 'GAME_EVENT_ID', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_NAME',
    
    # Game Context & Time
    'GAME_DATE', 'HTM', 'VTM', 'LOCATION', 'MATCH_TIME_df1', 'PERIOD_df1', 
    'PERIOD_TIME_REMAINING', 'FINAL_MARGIN',
    
    # Shot Details
    'GRID_TYPE', 'ACTION_TYPE', 'SHOT_TYPE', 'SHOT_ZONE_BASIC', 'SHOT_ZONE_AREA', 
    'SHOT_ZONE_RANGE', 'SHOT_DIST', 'LOC_X', 'LOC_Y', 
    'SHOT_CLOCK', 'DRIBBLES', 'TOUCH_TIME',
    
    # Outcomes
    'SHOT_MADE_FLAG',
    
    # Defender Data
    'CLOSEST_DEFENDER', 'CLOSEST_DEFENDER_PLAYER_ID', 'CLOSE_DEF_DIST',
    
    # Player Physicals (Note: ' Height' has a leading space in your raw list)
    'HEIGHT_INCHES', 'WEIGHT' 
]

df_cleaned = final_merged_df[columns_to_keep].copy()
df_cleaned = df_cleaned[df_cleaned['TOUCH_TIME'] > 0].copy()

rename_dict = {
    'MATCH_TIME_df1': 'MATCH_TIME',
    'PERIOD_df1': 'PERIOD',
}
df_cleaned = df_cleaned.rename(columns=rename_dict)
df_cleaned.dropna(inplace=True)

df_cleaned.to_csv('data/shotlog_2014-15_preprocessed.csv', index=False)

print(len(df_cleaned))
df_cleaned.head()

95892


,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,GAME_DATE,HTM,VTM,LOCATION,...,LOC_Y,SHOT_CLOCK,DRIBBLES,TOUCH_TIME,SHOT_MADE_FLAG,CLOSEST_DEFENDER,CLOSEST_DEFENDER_PLAYER_ID,CLOSE_DEF_DIST,HEIGHT_INCHES,WEIGHT
0,21400381,366,201196,Ramon Sessions,1610612758,Sacramento Kings,20141218,SAC,MIL,H,...,20.0,11.8,1,1.0,0,"Dudley, Jared",201162,0.8,75,190
2,21400381,483,202326,DeMarcus Cousins,1610612758,Sacramento Kings,20141218,SAC,MIL,H,...,6.0,5.4,2,2.3,1,"O'Bryant, Johnny",203948,2.9,82,270
3,21400571,215,202683,Enes Freedom,1610612762,Utah Jazz,20150113,UTA,GSW,H,...,0.0,0.1,0,0.8,0,"Thompson, Klay",202691,1.8,82,250
4,21400521,199,2564,Boris Diaw,1610612759,San Antonio Spurs,20150106,SAS,DET,H,...,7.0,10.9,1,1.1,1,"Jerebko, Jonas",201973,3.3,80,250
6,21400262,250,101112,Channing Frye,1610612753,Orlando Magic,20141202,GSW,ORL,A,...,219.0,17.9,0,2.2,1,"Green, Draymond",203110,2.4,84,255
